In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-05-27

@author: Juan Enrique López Marcos (CP)

@description: Jupyter Notebook creado para obtener la información publicada en portal TheHackerNews sobre noticias relacionadas con ciber delincuencia y posteriormente guardar la información formateada como archivo markdown para poder ser accedida desde una bóveda Obsidian.

'''

'\nCreated on 2024-05-27\n\n@author: Juan Enrique López Marcos (CP)\n\n@description: Jupyter Notebook creado para obtener la información publicada en portal TheHackerNews sobre noticias relacionadas con ciber delincuencia y posteriormente guardar la información formateada como archivo markdown para poder ser accedida desde una bóveda Obsidian.\n\n'

#### **Requerimientos**

In [2]:
import os
import re

import requests
from bs4 import BeautifulSoup

from datetime import datetime
import pandas as pd
import yaml


from dateutil.parser import parse

from stix2 import Filter, MemoryStore
import stix2

#### **Parámetros**

In [3]:
'''
Importante, se va a utilizar el método range, por lo que será contar desde 0 hasta el valor definido. Por ejemplo, si num_pages es 2 contará 2 url posteriores a la página principal, por lo que dispondremos de un total de 3 hojas de contenido.
'''
num_pages = 2

In [4]:
# Estructura de carpetas por fecha de publicación de la noticia
save_by_date = True

# Extensión del fichero final
save_as_md = True

In [5]:
# Parámetros opcionales
save_by_category_and_subcategory = False
save_by_unified_category = False
save_by_ttp = False

save_as_yaml = False

matrix = 'enterprise' # enterprise / ics / mobile

#### **Funciones**

In [6]:
def link_to_soup(link):
    '''
    Función cuyo cometido es peticionar a la url proporcionada como request y convertirlo en un objeto BeautifulSoup para su posterior tratamiento. Retornamos el propio objeto BeautifulSoup
    '''
    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    response = requests.get(link, headers=headers)
    if response.ok:
        return BeautifulSoup(response.text, 'html.parser')
    else:
        return False

In [7]:
def get_news_urls(main_bs4_page, num_scrap_pages, class_searched="blog-pager-older-link-mobile", link='https://thehackernews.com/'):
    '''
    Cuando entramos en la web de THN nos encontramos una página principal donde se listan las últimas n noticias junto con una ilustración y un breve texto. Con esta función obtenemos las url's propias de cada noticia para las n hojas pasadas como parámetro.
    '''
    pages = []
    pages.append(main_bs4_page)
    urls_temp = [link]
    posts_in_page = []
    urls_news = []
    check_url = r'https://thehackernews.com/202'

    # Mediante un bucle recorremos el número de paginas solicitadas para obtener los links de navegación entre páginas. Añadimos el contenido de toda la página al objeto pages
    for item in range(num_scrap_pages):
        # Buscamos, comenzando por la home page el link que te lleva a la siguiente url con noticias
        next_page_link = pages[item].find("a", class_=class_searched)['href']
        # Añadimos la url en una lista para tenerlo controlado
        urls_temp.append(next_page_link)
        # Generamos el objeto bs4 y lo añadimos
        pages.append(link_to_soup(next_page_link))
    # Filtramos el contenido con el objetivo de obtener las url de las noticias
    for page in pages:
        # Buscamos dentro del objeto bs4 la clase utilizada para publicar las noticias, importante que esto puede utilizarse para alojar anuncios y debemos hacer un filtro adicional
        posts_in_page = page.find_all("a", class_='story-link')
        for post in posts_in_page:
            # Filtramos solamente url que contengan la subcadena que las identifica como noticias
            if check_url.lower() in post['href'].lower():
                urls_news.append(post['href'])

    # Almacenadas las url que se van a scrapear
    urls_temp
    # Almacenado en un objeto todo el contenido de cada una de las urls
    pages
    return urls_news

In [8]:
def get_date_from_news(news):
    '''
    Obtención de la fecha de publicación de la noticia a partir del objeto bs4 que contiene la información completa de la noticia.
    '''
    # Obtener la fecha de publicación de la noticia
    spans = news.find_all('span')
    date_str = []
    # Iterar sobre los elementos <span> para encontrar aquellos con la clase "author"
    for span in spans:
        # Obtener la lista de clases del elemento <span>
        span_classes = span.get('class', [])
        # Verificar si "author" está en la lista de clases
        if 'author' in span_classes:
            date_str.append(span.get_text())
    try:
        # date_news = datetime.strptime(date_str[0], '%B %d, %Y')
        date_news = parse(date_str[0])
    except Exception as e:
        print(f'Error al intentar parsear la fecha: {e}')
    date_news = date_news.strftime('%d/%m/%Y')
    return date_news

In [9]:
def get_cat_from_news(news):
    '''
    Obtención de la categoría asignada por THN a la noticia a partir del objeto bs4 que contiene la información completa de la noticia.
    '''
    # Obtener la fecha de publicación de la noticia
    spans = news.find_all('span')
    cat_str = []
    # Iterar sobre los elementos <span> para encontrar aquellos con la clase "p-tags"
    try:
        for span in spans:
            # Obtener la lista de clases del elemento <span>
            span_classes = span.get('class', [])
            # Verificar si "p-tags" está en la lista de clases
            if 'p-tags' in span_classes:
                cat_str.append(str(span.get_text()).replace('_', ' '))
    except:
        cat_str=''
    
    return cat_str

In [10]:
def get_body_and_links_from_news(news):
    '''
    Obtención del cuerpo y links en la noticia a partir del objeto bs4 que contiene la información completa.
    '''
    body = []
    news_links = []
    for p in news.find_all('p'):
        body.append(str(p.get_text()))
        links = p.find_all('a')
        for link in links:
            news_links.append(str(link.get('href')))
    return news_links, body

In [11]:
def get_news_info(news_url_list):
    '''
    Función que, dada una lista de urls de las noticias recopiladas en THN retorna una lista de diccionarios que contienen la información de cada noticia. Retorna la lista de diccionarios con la información
    '''
    news_info = []
    for n in news_url_list:
        n_info = {}
        n_bs4 = link_to_soup(n)
        n_title = str(n_bs4.title.string)
        print(n)
        n_date = get_date_from_news(n_bs4)
        n_cat = get_cat_from_news(n_bs4)
        n_links, n_body = get_body_and_links_from_news(n_bs4)
        
        n_info['title'] = n_title
        n_info['date'] = n_date
        n_info['url_news'] = n
        n_info['thn_category_and_subcategory'] = n_cat
        n_info['links'] = n_links
        n_info['body'] = n_body
        news_info.append(n_info)

    return news_info

In [12]:
def format_name(item):
    '''
    Función creada para la eliminación de caracteres extraños en los elementos str de una lista. Retorna la lista formateada.
    '''
    if isinstance(item, list):
        item = '_'.join(map(str, item))
    item = re.sub(r'[\\/:*?"<>|]', '', item)
    item = re.sub(r'\s+', '_', item)  # Reemplazar espacios por guiones bajos
    item = re.sub(r'[^a-zA-Z0-9_\-]', '', item)  # Eliminar cualquier otro carácter no alfanumérico
    return item

In [13]:
def format_name_cat(item):
    '''
    Función específica para la obtención y formateo de la categoría a partir del etiquetado global asignada por THN. THN asigna generalmente un etiquetado global conformado por lo que nosotros hemos identificado como catgoría-subcatgoría. En esta función únicamente se retorna la categoría o primera etiqueta asignada. Retorna la lista formateada.
    '''
    # Convertimos en string en el caso de pasar una lista
    if isinstance(item, list):
        item = '_'.join(map(str, item))
    # En primer lugar nos asguramos de que no exista ninguna barra baja
    item = item.replace('_', ' ')
    # Reemplazamos el caracter que separa categoria de dsubcategoria por barra baja
    item = item.replace(' / ', '_')
    # Reemplazar espacios por guiones
    item = item.replace(' ', '-') 
    # Reemplazamos resto de caracteres no alfanumérico
    item = re.sub(r'[^a-zA-Z0-9_\-]', '', item)
    return item

In [14]:
def create_header_properties(dictionary):
    '''
    Con el objetivo de integrar en el fichero markdown final las propiedades por las que poder filtrar en obsidian. Mediante la función se mapean una serie de características y se retorna una cadena para poder añadir al principio del documento .md.
    '''
    cat_subcat = dictionary.get('thn_category_and_subcategory', [])[:1] or []
    cat_subcat = [subitem.strip() for str in cat_subcat for subitem in str.split('/')]
    # cat_subcat = [cat.strip() for cat in cat_subcat]
    cat, subcat = (cat_subcat + ['', ''])[:2]
    
    header = f"""---
CP Source: The Hacker News
CP Execution date:  {datetime.now().strftime('%Y-%m-%d')}
Headline: {'"' + dictionary.get('title') + '"'}
Date: {datetime.strptime(dictionary.get('date'), '%d/%m/%Y').strftime('%Y-%m-%d')}
Category: {'"'+cat+'"'}
Sub-Category: {'"'+subcat+'"'}
---
"""
    return header

In [15]:
def write_dict_to_md(dictionary, file_path):
    '''
    Función de escritura y guardado del contenido de la noticia como archivo .markdown. 
    '''
    try:
        with open(file_path, 'w', encoding='utf-8') as file:
            header_properties = create_header_properties(dictionary)
            file.write(header_properties+ "\n")
            for key, value in dictionary.items():
                value = str(value).replace('[', '').replace(']', '')
                file.write(f"**{key}**\n{value}\n\n")
        print(f"Archivo .md guardado en: {file_path}")
    except (OSError, IOError) as e:
        print(f"Error al escribir en el archivo {file_path}: {e}")

In [16]:
def print_oldest_and_most_recent_notice(dict_news):
    '''
    Función encargada de mostrar la fecha de la noticia más reciente y la noticia más antigua a partir del diccionario que contiene la información de las noticias scrapeadas.
    '''
    from datetime import datetime
    dates = [datetime.strptime(dic['date'], '%d/%m/%Y') for dic in dict_news]
    oldest = min(dates)
    most_recent = max(dates)
    oldest = oldest.strftime('%d/%m/%Y')
    most_recent = most_recent.strftime('%d/%m/%Y')
    print('Noticia más reciente: '+most_recent)
    print('Noticia más antigua: '+oldest)

In [17]:
def get_count_files_from_path(main_folder, col_name):
    '''
    Función encargada de construir un df que contendrá por un lado el nombre de la subcarpeta y por otro lado el número de archivos identificados en esa. Requiere un path así como un nombre para la columna de recuento.
    '''
    name_folders = []
    count_files = []
    for path, subcarpetas, archivos in os.walk(main_folder):
        if path == main_folder:
            continue
        name_folder = os.path.basename(path)
        # Contar el número de archivos en la carpeta actual
        count_file = len(archivos)
        # Agregar el nombre de la carpeta y el conteo a las listas
        name_folders.append(name_folder)
        count_files.append(count_file)
    # Crear un DataFrame de pandas
    df = pd.DataFrame({col_name: name_folders, 'items': count_files})
    # df['cat_subcat'] = df['cat_subcat'].apply(lambda x: x if '_' in x else f"{x}_")
    return df

##### **Funciones opcionales**

Aquí se detallan otras funciones no requeridas para el funcionamiento principal del código.

In [18]:
def split_list_by_hyphen(items):
    '''
    Función encargada de retornar una lista con las etiquetas asignadas como categoría y subcategoría a la noticia.
    '''
    result = []
    for item in items:
        item = item.replace('-', ' ')
        if '/' in item:
            parts = item.split('/')
            result.extend([part.strip() for part in parts])  # Elimina espacios en blanco alrededor
        else:
            result.append(item.strip())  # Elimina espacios en blanco alrededor
    return result

In [19]:
def get_unified_cat(item):
    '''
    Función específica para unificar el formateo de las etiquetas asignadas.
    '''
    cats = split_list_by_hyphen(item)
    cats = [elemento.strip() for elemento in cats]
    cats = [re.sub(r'[^a-zA-Z0-9_\-, ]', '', cat) for cat in cats]
    return cats

In [20]:
def modify_string(s):
    '''
    Breve función para formatear posibles casos en los que se hayan asignado más de una subcategoría.
    '''
    if s.count('_') == 2:
        # Reemplazar el segundo "_" encontrado
        parts = s.split('_', 2)
        return parts[0] + '_' + parts[1] + parts[2]
    return s

In [21]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    stix_json = requests.get(url).json()
    return MemoryStore(stix_data=stix_json["objects"])

In [22]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [23]:
def find_techniques(texto, techniques_list):
    '''
    Función encargada de buscar en un texto facilitado los elementos contenidos en la lista de técnicas. Retorna una lista con las técnicas encontradas.
    '''
    if isinstance(texto, str):
        found = []
        for item in techniques_list:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

In [24]:
class NoAliasDumper(yaml.SafeDumper):
    '''
    La clase NoAliasDumper es una extensión de yaml.SafeDumper que sobrescribe el método ignore_aliases para siempre devolver True. Esto asegura que los datos se serialicen sin usar referencias alias en el formato YAML.
    '''
    def ignore_aliases(self, data):
        return True


def write_dict_to_yaml(news, filename):
    '''
    Función de escritura y guardado del contenido de la noticia como archivo .yaml. 
    '''
    try:
        # Ordenamos las claves del diccionario
        ordered_keys = ['title', 'url_news', 'thn_category_and_subcategory', 'date', 'links', 'body']
        ordered_news = {key: news[key] for key in ordered_keys if key in news}
        
        with open(filename, 'w') as file:
            yaml.dump(ordered_news, file, default_flow_style=False, allow_unicode=True, Dumper=NoAliasDumper, sort_keys=False)
        print(f"Noticia escrita en {filename}")
    except yaml.YAMLError as e:
        print(f"Error al serializar el archivo YAML: {e}")
    except IOError as e:
        print(f"Error de I/O al escribir el archivo: {e}")

#### **Ejecución principal**

In [25]:
url='https://thehackernews.com/'
home_page = link_to_soup(url)

In [26]:
# Obtención de las url de noticias 
news_list = get_news_urls(home_page, num_pages, class_searched="blog-pager-older-link-mobile")
print(f'Se han obtenido {str(len(news_list))} noticias')

Se han obtenido 30 noticias


In [27]:
# Obtenemos la información de las noticias y la almacenamos en una lista de diccionarios
news_info = get_news_info(news_list)

https://thehackernews.com/2024/07/how-mfa-failures-are-fueling-500-surge.html
https://thehackernews.com/2024/07/new-intel-cpu-vulnerability-indirector.html
https://thehackernews.com/2024/07/metas-pay-or-consent-approach-faces-eu.html
https://thehackernews.com/2024/07/chinese-hackers-exploiting-cisco.html
https://thehackernews.com/2024/07/australian-man-charged-for-fake-wi-fi.html
https://thehackernews.com/2024/07/critical-flaws-in-cocoapods-expose-ios.html
https://thehackernews.com/2024/07/caprarat-spyware-disguised-as-popular.html
https://thehackernews.com/2024/06/the-secrets-of-hidden-ai-training-on.html
https://thehackernews.com/2024/07/indian-software-firms-products-hacked.html
https://thehackernews.com/2024/07/end-to-end-secrets-security-making-plan.html
https://thehackernews.com/2024/07/new-openssh-vulnerability-could-lead-to.html
https://thehackernews.com/2024/07/juniper-networks-releases-critical.html
https://thehackernews.com/2024/06/the-secrets-of-hidden-ai-training-on.html
h

**Guardado por fecha**

In [28]:
if save_by_date:
    for n in news_info:
        if len(n.get('date')) != '':
            folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_date', datetime.strptime(n.get('date'), '%d/%m/%Y').strftime('%Y%m%d'))
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_date', 'no-date')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_yaml:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
            write_dict_to_yaml(n, filename)
        elif save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)

Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\TheHackerNews\save_by_date\20240702\how_mfa_failures_are_fueling_a_500_surge_in_ransomware_losses.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\TheHackerNews\save_by_date\20240702\new_intel_cpu_vulnerability_indirector_exposes_sensitive_data.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\TheHackerNews\save_by_date\20240702\metas_pay_or_consent_approach_faces_eu_competition_rules_scrutiny.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\TheHackerNews\save_by_date\20240702\chinese_hackers_exploiting_cisco_switches_zero-day_to_deliver_malware.md
Archivo .md guardado en: c:\Users\jelopez\Documents\CyberProof\python\develop\get_cti_information\outputs\TheHackerNews\save_by_date\20240702\australian_man_cha

****Revisión resultados - fecha de la noticia****

In [29]:
if save_by_date:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_date')
    results = get_count_files_from_path(main_folder, 'date')
    display(results.sort_values(by='items', ascending=False).head(10))

,date,items
2,20240628,7
1,20240627,6
4,20240701,6
5,20240702,5
0,20240626,2
3,20240629,1


#### **Opcional**

Se ha desarrollado código adicional para una posible clasificación por categoria y subcategoria, unificacion de categorias o por identificación de TTP.

##### **Guardado de los archivos por categoría y subcategoría THN**

In [30]:
if save_by_category_and_subcategory:
    for n in news_info:
        if len(n.get('thn_category_and_subcategory'))==1:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews','save_by_category_and_subcategory', format_name_cat(n.get('thn_category_and_subcategory')).lower())
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_category_and_subcategory', 'no-category')
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
        if save_as_yaml:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
            write_dict_to_yaml(n, filename)
        elif save_as_md:
            filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
            write_dict_to_md(n, filename)
    print_oldest_and_most_recent_notice(news_info)

**Revisión resultados - categoría y subcategoría THN**

In [31]:
if save_by_category_and_subcategory:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_category_and_subcategory')
    result = get_count_files_from_path(main_folder, 'cat_subcat')
    result = result.sort_values(by='items', ascending=False)
    #Filtramos la carpeta outputs
    result = result[result['cat_subcat']!='outputs']
    result['cat_subcat'] = result['cat_subcat'].apply(modify_string)
    # # Añadimos una barra baja al final de todos los string que no contengan subcategoria
    result['cat_subcat'] = result['cat_subcat'].apply(lambda x: x if '_' in x else f"{x}_")
    result[['cat', 'subcat']] = result['cat_subcat'].str.split('_', expand=True)
    result.sort_values(by='items', ascending=False).head(5)

##### **Guardado de los archivos por categoría unificada THN**

In [32]:
if save_by_unified_category:
    for n in news_info:
        if len(n.get('thn_category_and_subcategory'))==1:
            cats = get_unified_cat(n.get('thn_category_and_subcategory'))
            for cat in cats:
                folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_unified_category', cat.lower())
                if not os.path.exists(folder_name):
                    os.makedirs(folder_name)
                if save_as_yaml:
                    filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
                    write_dict_to_yaml(n, filename)
                elif save_as_md:
                    filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
                    write_dict_to_md(n, filename)
        else:
            folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_unified_category', 'no-category')
            if not os.path.exists(folder_name):
                os.makedirs(folder_name)
            if save_as_yaml:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
                write_dict_to_yaml(n, filename)
            elif save_as_md:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
                write_dict_to_md(n, filename)
    print_oldest_and_most_recent_notice(news_info)
    

**Revisión resultados - categoría unificada THN**

In [33]:
if save_by_unified_category:
    main_folder = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_unified_category')
    results = get_count_files_from_path(main_folder, 'save_by_unified_category') # Chequear la introduccion de la subcarpeta 'TheHackerNews',
    display(results.sort_values(by='items', ascending=False).head(10))

##### **Identifiación y guardado por TTP's en la noticia**

In [34]:
if save_by_ttp:
    src = get_data_from_branch(matrix)
    techniques = get_list_techniques_from_stix2(src)
    ttp_list = []
    for n in news_info:
        unified_body = ' '.join(n['body'])
        ttp_finded = find_techniques(unified_body, techniques)
        ttp_list.append(ttp_finded)
    ttp_list_unique_set = set(ttp_list)
    ttp_list_unique = list(ttp_list_unique_set)
    # TTP's unicas encontradas
    ttp_list_unique

**Guardado por TTP en la noticia**

In [35]:
if save_by_ttp:
    if len(ttp_list_unique) > 1:
        for n in news_info:
            if len(n.get('thn_category_and_subcategory'))==1:
                folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', 'save_by_ttp', matrix, format_name_cat(n.get('thn_category_and_subcategory')).lower())
            else:
                folder_name = os.path.join(os.getcwd(), 'outputs', 'thehackernews', matrix, 'no-category')
            if not os.path.exists(folder_name):
                os.makedirs(folder_name)
            if save_as_yaml:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.yml')
                write_dict_to_yaml(n, filename)
            elif save_as_md:
                filename = os.path.join(folder_name, format_name(n.get("title").lower())+'.md')
                write_dict_to_md(n, filename)
    elif len(ttp_list_unique) <= 1:
        print("No se han obtenido TTP's de las noticias por lo que no se ha procedido al guardado.")